<a href="https://colab.research.google.com/github/whit7990/usports-fb-xml-check/blob/main/XML_Play_Validator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [145]:
#@title 1. Upload XML File
from google.colab import files

print("Please upload your PrestoSports Stat Crew XML file:")
uploaded = files.upload()
xml_filename = list(uploaded.keys())[0]
print(f"Successfully loaded: {xml_filename}\n")

Please upload your PrestoSports Stat Crew XML file:


Saving boxscore_20260907_1249.xml to boxscore_20260907_1249 (8).xml
Successfully loaded: boxscore_20260907_1249 (8).xml



In [146]:
#@title 2. Parse XML and Build Roster Dictionary
import xml.etree.ElementTree as ET

# Parse XML root
tree = ET.parse(xml_filename)
root = tree.getroot()

roster = {}
for team_tag in ['visiting-team', 'home-team', 'team']:
    for team in root.findall(f".//{team_tag}"):
        for player in team.findall('.//player'):
            pid = player.get('id') or player.get('uni')
            if pid:
                roster[pid] = {
                    'uni': player.get('uni'),
                    'pos': player.get('pos', '').upper(),
                    'name': player.get('name', '')
                }

print(f"Roster loaded successfully. Total players indexed: {len(roster)}")

Roster loaded successfully. Total players indexed: 52


In [147]:
#@title 3. Audit Plays and Run QC Rules
import re

# Standard Canadian Penalty Yardage & Code Reference Map
CANADIAN_PENALTIES = {
    'OC': (10, 'Objectionable Conduct'),
    'UR': (15, 'Unnecessary Roughness'),
    'CHD': (15, 'Chop Block / Delayed Knee Block'),
    'OS': (5, 'Offside'),
    'IP': (5, 'Illegal Procedure'),
    'HO': (10, 'Holding'),
    'DG': (10, 'Delay of Game'),
    'TC': (5, 'Time Count Violation'),
    '12': (10, 'Too Many Men on the Field'),
    'UF': None
}

errors = []
warnings = []
pass_rec_results = []
total_plays = 0

def extract_play_number(play_tag):
    """
    Extracts the 3rd number strictly from the playid attribute value.
    Example: playid='FBK-2026-Q1-P12' -> extracts '12' as the 3rd number.
    """
    play_id_val = play_tag.get('playid', '')
    if play_id_val:
        numbers = re.findall(r'\d+', play_id_val)
        if len(numbers) >= 3:
            return numbers[2]
    return "N/A"

def extract_starting_yardline(text):
    """
    Extracts and normalizes the starting yard line designation to V## or H## format.
    """
    matches = re.findall(r'([A-Z]+)(\d{1,2})', text)
    if matches:
        team, yard = matches[0]
        prefix = "H" if "home" in team.lower() or team not in ["GUELPH", "VISITOR"] else "V"
        return f"{prefix}{yard.zfill(2)}"
    return "H00"

# Audit Plays loop (iterating through each quarter)
for qtr in root.findall('.//qtr'):
    qtr_num = qtr.get('number', '1')
    plays_in_qtr = qtr.findall('play')
    num_plays = len(plays_in_qtr)

    for i in range(num_plays):
        play = plays_in_qtr[i]
        total_plays += 1
        raw_play_num = extract_play_number(play)
        clock = play.get('clock', '')
        text = play.get('text', '')
        text_lower = text.lower()
        score_flag = play.get('score', 'N')

        start_yl = extract_starting_yardline(text)
        play_num = f"{raw_play_num} ({start_yl})"

        play_desc = f"Q{qtr_num} {clock if clock else 'No Clock'} - {text}"
        is_noplay = 'no play' in text_lower or 'noplay' in text_lower

# --- CHECK 1: Clock & Possession Change Checks ---
        if not is_noplay:
            is_scoring = (score_flag == 'Y' or play.find('scores') is not None)
            is_touchdown = 'touchdown' in text_lower
            is_pat_convert = any(term in text_lower for term in ['kick attempt good', 'convert good', 'point attribute'])

            # A turnover/kick is a possession change, but a pure touchdown play is a score (possession changes on kickoff)
            is_actual_possession_change = any(term in text_lower for term in ['intercept', 'fumble lost', 'turnover on downs', 'punt', 'kickoff return', 'field goal attempt', 'missed'])

            should_check_clock = (is_scoring and not is_touchdown) or (is_actual_possession_change and not is_pat_convert)

            if should_check_clock:
                if not clock:
                    errors.append({
                        'play_num': play_num,
                        'details': play_desc,
                        'issue': "Clock time is missing on a scoring play or change of possession/kick sequence."
                    })
                else:
                    next_clock = plays_in_qtr[i + 1].get('clock', '') if i + 1 < num_plays else ''
                    prev_clock = plays_in_qtr[i - 1].get('clock', '') if i - 1 >= 0 else ''

                    if clock == prev_clock and (not next_clock or clock == next_clock):
                        errors.append({
                            'play_num': play_num,
                            'details': play_desc,
                            'issue': "Clock time remains static (unchanged across adjacent plays) on a major possession change or kick sequence."
                        })

            if clock:
                try:
                    parts = clock.split(':')
                    minutes = int(parts[0])
                    if minutes > 15 or minutes < 0:
                        errors.append({
                            'play_num': play_num,
                            'details': play_desc,
                            'issue': f"Clock value {clock} exceeds 15:00 regulation limit or is negative."
                        })
                except ValueError:
                    pass

        # --- CHECK 2: Penalty Code & Yardage Validation (with Inside-15 Yardline Note) ---
        numeric_yl = None
        yl_match = re.search(r'[A-Z]+(\d{1,2})', text)
        if yl_match:
            try:
                numeric_yl = int(yl_match.group(1))
            except ValueError:
                pass

        is_inside_fifteen = (numeric_yl is not None and 1 <= numeric_yl <= 15)

        for pn in play.findall('.//p_pn'):
            accept_status = pn.get('intent', pn.get('accept', '')).lower()
            if accept_status in ['no', 'false', 'declined'] or 'declined' in text_lower:
                continue

            code = pn.get('code', '').upper()
            try:
                yards_assessed = int(pn.get('yards', 0))
            except (ValueError, TypeError):
                yards_assessed = 0

            if code in ['PF', 'UC']:
                errors.append({
                    'play_num': play_num,
                    'details': play_desc,
                    'issue': f"American penalty code/term '{code}' used. Must use Canadian terminology."
                })
            elif code in CANADIAN_PENALTIES and CANADIAN_PENALTIES[code]:
                std_yards, penalty_name = CANADIAN_PENALTIES[code]
                if isinstance(std_yards, int) and std_yards != 0 and yards_assessed != std_yards:
                    if is_inside_fifteen:
                        warnings.append({
                            'play_num': play_num,
                            'details': play_desc,
                            'issue': f"Penalty {code} ({penalty_name}) assessed {yards_assessed} yards (standard is {std_yards}). Note: Play started inside the 15-yard line ({start_yl})."
                        })
                    else:
                        warnings.append({
                            'play_num': play_num,
                            'details': play_desc,
                            'issue': f"Penalty {code} ({penalty_name}) assessed {yards_assessed} yards; standard Canadian distance is {std_yards} yards."
                        })

        # --- CHECK 5: Large Yardage Loss Detection (>= 20 yards) ---
        loss_patterns = [
            r'loss of (\d{2,})',
            r'sacked for (-?\d{2,})',
            r'for loss of (\d{2,})',
            r'\b(-\d{2,})\b'
        ]

        large_loss_found = False
        loss_val_detected = 0
        for pattern in loss_patterns:
            match = re.search(pattern, text_lower)
            if match:
                try:
                    val = abs(int(match.group(1)))
                    if val >= 20:
                        large_loss_found = True
                        loss_val_detected = val
                        break
                except ValueError:
                    pass

        for subelem in play.iter():
            for attr in ['loss', 'sackyds', 'yds']:
                val_str = subelem.get(attr, '')
                if val_str:
                    try:
                        v = int(val_str)
                        if v <= -20 or (attr in ['loss', 'sackyds'] and v >= 20):
                            large_loss_found = True
                            loss_val_detected = abs(v)
                    except ValueError:
                        pass

        if large_loss_found:
            warnings.append({
                'play_num': play_num,
                'details': play_desc,
                'issue': f"Large yardage loss detected (approx. {loss_val_detected} yards). Please verify play metrics."
            })

# --- CHECK 6: Unauthorized TEAM / TM Stat Attribution & Team Rush Losses ---
        is_team_safety = 'safety' in text_lower and ('team' in text_lower or 'tm' in text_lower)
        play_flagged_for_team_stat = False

        if not is_team_safety:
            # 1. Check text-based team attributions (e.g., tackles by (TEAM; TEAM))
            if re.search(r'\((TEAM|TM);\s*(TEAM|TEAM)\)', text, re.IGNORECASE):
                errors.append({
                    'play_num': play_num,
                    'details': play_desc,
                    'issue': "TEAM or TM should not be given stats: generic team entity '(TEAM; TEAM)' found in play text attribution."
                })
                play_flagged_for_team_stat = True

            # 2. Check all sub-elements for TEAM/TM stats (only if not already flagged)
            if not play_flagged_for_team_stat:
                for subelem in play.iter():
                    s_name = subelem.get('name', '').upper()
                    s_uni = subelem.get('uni', '').upper()

                    if s_name in ['TEAM', 'TM'] or s_uni in ['TEAM', 'TM']:
                        has_ff = (subelem.get('ff', '').upper() == 'Y')
                        has_stats = any(subelem.find(tag) is not None for tag in ['rush', 'pass', 'rcv', 'tackle', 'fum', 'forced'])
                        has_inline_attr = any(subelem.get(attr) not in [None, '0', '', 'N'] for attr in ['ff', 'forced', 'fumble_forced', 'tackles'])

                        if has_ff or has_stats or has_inline_attr or subelem.tag in ['p_tk', 'p_fum']:
                            errors.append({
                                'play_num': play_num,
                                'details': play_desc,
                                'issue': f"TEAM or TM should not be given stats: statistics incorrectly credited to generic entity '{s_name or s_uni}' via tag <{subelem.tag}>."
                            })
                            play_flagged_for_team_stat = True
                            break

            # 3. Check standalone fumble elements for TEAM/TM
            if not play_flagged_for_team_stat:
                for fum_elem in play.findall('.//fum'):
                    fum_forced_by = fum_elem.get('forced_by', '') or fum_elem.get('ff_by', '')
                    if fum_forced_by.upper() in ['TEAM', 'TM']:
                        errors.append({
                            'play_num': play_num,
                            'details': play_desc,
                            'issue': "TEAM or TM should not be given stats: forced fumble statistic incorrectly attributed to TEAM/TM instead of an individual player."
                        })
                        play_flagged_for_team_stat = True
                        break

            # 4. Check rush for loss credited to team
            if not play_flagged_for_team_stat and 'rush' in text_lower and ('loss' in text_lower or 'sacked' in text_lower or '(-' in text_lower):
                for sub in play.findall('.//rush'):
                    try:
                        yds = int(sub.get('yds', 0))
                        loss_val = int(sub.get('loss', 0))
                        if yds < 0 or loss_val > 0:
                            if 'team' in str(sub.attrib).lower() or 'tm' in text_lower:
                                errors.append({
                                    'play_num': play_num,
                                    'details': play_desc,
                                    'issue': "TEAM or TM should not be given stats: rush for a loss incorrectly credited as a team-level stat instead of an individual player."
                                })
                                break
                    except ValueError:
                        pass

# --- PASSING VS. RECEIVING YARDS VALIDATION (Per Team) ---
team_nodes = root.findall('.//team')
team_stats = {}

for i, team in enumerate(team_nodes):
    vh_attr = team.get('vh', '').upper()
    if vh_attr == 'H' or (i == 1 and not vh_attr):
        team_label = "Home"
    elif vh_attr == 'V' or (i == 0 and not vh_attr):
        team_label = "Visitor"
    else:
        team_label = team.get('name', f"Team {i+1}")

    team_stats[id(team)] = {
        'label': team_label,
        'team_node': team,
        'pass_yds': 0,
        'rcv_yds': 0
    }

player_to_team_id = {}
for t_id, data in team_stats.items():
    t_node = data['team_node']
    for player in t_node.iter('player'):
        player_to_team_id[id(player)] = t_id

for player in root.iter('player'):
    t_id = player_to_team_id.get(id(player))
    if t_id and t_id in team_stats:
        pass_tag = player.find('pass')
        if pass_tag is not None:
            try:
                team_stats[t_id]['pass_yds'] += int(pass_tag.get('yds', 0))
            except ValueError:
                pass

        for rcv_tag_name in ['rcv', 'receiving']:
            rcv_tag = player.find(rcv_tag_name)
            if rcv_tag is not None:
                try:
                    team_stats[t_id]['rcv_yds'] += int(rcv_tag.get('yds', 0))
                except ValueError:
                    pass

for stats in team_stats.values():
    t_label = stats['label']
    p_yds = stats['pass_yds']
    r_yds = stats['rcv_yds']

    if p_yds == r_yds:
        pass_rec_results.append(f"* **{t_label}**: [PASS] Passing Yards ({p_yds}) match Receiving Yards ({r_yds}).")
    else:
        pass_rec_results.append(f"* **{t_label}**: [ERROR] Passing Yards ({p_yds}) do not match Receiving Yards ({r_yds}).")

# --- PLAYER STAT ENTRY LOGIC VALIDATION ---
for player in root.iter('player'):
    name = player.get('name', 'Unknown')
    uni = player.get('uni', 'N/A')

    if name.upper() in ['TEAM', 'TM'] or uni.upper() in ['TEAM', 'TM']:
        continue

    pass_tag = player.find('pass')
    if pass_tag is not None:
        try:
            att = int(pass_tag.get('att', 0))
            comp = int(pass_tag.get('comp', 0))
            if comp > att:
                errors.append({
                    'play_num': f"Player #{uni} ({name})",
                    'details': f"Passing Stats Record -> Attempts: {att}, Completions: {comp}",
                    'issue': f"Completions ({comp}) exceed passing attempts ({att})."
                })
            if att < 0 or comp < 0:
                errors.append({
                    'play_num': f"Player #{uni} ({name})",
                    'details': f"Passing Stats Record -> Attempts: {att}, Completions: {comp}",
                    'issue': 'Negative passing attempts or completions found.'
                })
        except ValueError:
            pass

    rush_tag = player.find('rush')
    if rush_tag is not None:
        try:
            att = int(rush_tag.get('att', 0))
            if att < 0:
                errors.append({
                    'play_num': f"Player #{uni} ({name})",
                    'details': f"Rushing Stats Record -> Attempts: {att}",
                    'issue': f"Negative rushing attempts ({att}) found."
                })
        except ValueError:
            pass

print(f"Audit complete. Reviewed {total_plays} plays and all player stat records.")

Audit complete. Reviewed 169 plays and all player stat records.


In [148]:
#@title 4. Output Structured Report
from IPython.display import display, Markdown

def parse_top_to_seconds(top_str):
    """Converts a TOP string (e.g., '28:15') into total seconds."""
    if not top_str or ':' not in top_str:
        return 0
    try:
        parts = top_str.split(':')
        return int(parts[0]) * 60 + int(parts[1])
    except (ValueError, IndexError):
        return 0

def format_seconds_to_mmss(total_seconds):
    """Converts total seconds back into MM:SS format."""
    m = total_seconds // 60
    s = total_seconds % 60
    return f"{m:02d}:{s:02d}"

# --- TIME OF POSSESSION (TOP) COMBINED GAME TOTAL CHECK ---
quarters_found = root.findall('.//qtr')
max_qtr = 0
for qtr in quarters_found:
    try:
        q_num = int(qtr.get('number', '0'))
        if q_num > max_qtr:
            max_qtr = q_num
    except ValueError:
        pass

expected_total_seconds = 1800 if max_qtr <= 2 else 3600
expected_label = "30 minutes (half)" if max_qtr <= 2 else "60 minutes (full game)"

combined_top_seconds = 0
team_top_records = []

misc_nodes = root.findall('.//misc')
for i, misc in enumerate(misc_nodes):
    vh_attr = misc.get('vh', '').upper()
    if vh_attr == 'H' or (i == 1 and not vh_attr):
        team_label = "Home"
    elif vh_attr == 'V' or (i == 0 and not vh_attr):
        team_label = "Visitor"
    else:
        team_label = misc.get('team', f"Team {i+1}")

    top_val = misc.get('top', '')
    if top_val:
        secs = parse_top_to_seconds(top_val)
        combined_top_seconds += secs
        team_top_records.append(f"{team_label}: {top_val}")

diff_seconds = abs(combined_top_seconds - expected_total_seconds)
combined_str = format_seconds_to_mmss(combined_top_seconds)

# Generate Structured Markdown Output
markdown_output = []
markdown_output.append("=" * 60)
markdown_output.append("### CANADIAN FOOTBALL XML QUALITY CONTROL REPORT")
markdown_output.append("=" * 60)

# TOP Section
markdown_output.append("\n## TIME OF POSSESSION")
if team_top_records:
    markdown_output.append(f"* **Team Breakdown**: " + " | ".join(team_top_records))
    markdown_output.append(f"* **Combined Game Total**: {combined_str} (Expected: {expected_label})")
    if diff_seconds <= 5:
        markdown_output.append("* **Status**: [TOP PASS] Combined Time of Possession matches expected duration.")
    else:
        markdown_output.append(f"* **Status**: [ERROR] Combined Time of Possession ({combined_str}) does not equal expected {expected_label}.")
else:
    markdown_output.append("* **Status**: [TOP NOTICE] No team 'misc' statistics nodes with 'top' attributes found in XML.")

# Passing vs Receiving Section
markdown_output.append("\n## PASSING VS. RECEIVING YARDS")
if pass_rec_results:
    for res in pass_rec_results:
        markdown_output.append(res)
else:
    markdown_output.append("* **Status**: [NOTICE] No team passing/receiving statistics nodes found in XML.")

markdown_output.append(f"\n## ERROR ({len(errors)})")
if not errors:
    markdown_output.append("* No errors found.")
else:
    for err in errors:
        markdown_output.append(f"* **Play #**: {err['play_num']}")
        markdown_output.append(f"  * **Play Details**: {err['details']}")
        markdown_output.append(f"  * **Issue Identified**: {err['issue']}")

markdown_output.append(f"\n## WARNING ({len(warnings)})")
if not warnings:
    markdown_output.append("* No warnings found.")
else:
    for warn in warnings:
        markdown_output.append(f"* **Play #**: {warn['play_num']}")
        markdown_output.append(f"  * **Play Details**: {warn['details']}")
        markdown_output.append(f"  * **Issue Identified**: {warn['issue']}")

markdown_output.append("\n" + "-" * 60)
markdown_output.append(f"**Total plays reviewed**: {total_plays}")

# Join and display using IPython Markdown display tool
full_markdown_str = "\n".join(markdown_output)
display(Markdown(full_markdown_str))

============================================================
### CANADIAN FOOTBALL XML QUALITY CONTROL REPORT
============================================================

## TIME OF POSSESSION
* **Team Breakdown**: Visitor: 40:13 | Home: 28:10
* **Combined Game Total**: 68:23 (Expected: 60 minutes (full game))
* **Status**: [ERROR] Combined Time of Possession (68:23) does not equal expected 60 minutes (full game).

## PASSING VS. RECEIVING YARDS
* **Visitor**: [PASS] Passing Yards (191) match Receiving Yards (191).
* **Home**: [PASS] Passing Yards (242) match Receiving Yards (242).

## ERROR (5)
* **Play #**: 25 (V01)
  * **Play Details**: Q1 10:00 - Marcus Reypa field goal attempt from 45 MISSED, kick to GUELPH-15, clock 10:00, Anakin Guthrie return 16 yards to the GUELPH01 (Josh Kibbee), clock 10:00.
  * **Issue Identified**: Clock time remains static (unchanged across adjacent plays) on a major possession change or kick sequence.
* **Play #**: 53 (H51)
  * **Play Details**: Q2 10:06 - Joey Edlington rush for no gain to the LAURIER51, fumble forced by TEAM, fumble by Joey Edlington recovered by LAURIER Marcus Reypa at LAURIER20, clock 10:06.
  * **Issue Identified**: TEAM or TM should not be given stats: statistics incorrectly credited to generic entity 'TEAM' via tag <p_tk>.
* **Play #**: 57 (H00)
  * **Play Details**: Q2 09:29 - Trey Thompson kick attempt good, clock 09:29.
  * **Issue Identified**: Clock time remains static (unchanged across adjacent plays) on a major possession change or kick sequence.
* **Play #**: 98 (V37)
  * **Play Details**: Q3 12:01 - Will Russell pass intercepted by Tyson Harker at the GUELPH37, Tyson Harker return 1 yards to the GUELPH38 (TEAM), clock 12:01.
  * **Issue Identified**: TEAM or TM should not be given stats: statistics incorrectly credited to generic entity 'TEAM' via tag <p_tk>.
* **Play #**: 140 (V41)
  * **Play Details**: Q4 13:53 - Tayshaun Jackson rush for 8 yards to the GUELPH41 (TEAM), clock 13:53.
  * **Issue Identified**: TEAM or TM should not be given stats: statistics incorrectly credited to generic entity 'TEAM' via tag <p_tk>.

## WARNING (2)
* **Play #**: 84 (V21)
  * **Play Details**: Q2 01:25 - Marcus Reypa kickoff 70 yards to the GUELPH-5, Alex Poitevien return 26 yards to the GUELPH21 (Darcy Winsor), PENALTY LAURIER unnecessary roughness 20 yards to the GUELPH41, PENALTY LAURIER offside 0 yards to the GUELPH41, clock 01:25.
  * **Issue Identified**: Penalty OS (Offside) assessed 0 yards; standard Canadian distance is 5 yards.
* **Play #**: 161 (V01)
  * **Play Details**: Q4 00:55 - Tayshaun Jackson rush for 1 yard to the GUELPH01 (Lachlan Burke), PENALTY GUELPH 12 men on the field 1 yards to the GUELPH01, NO PLAY, clock 00:55.
  * **Issue Identified**: Penalty 12 (Too Many Men on the Field) assessed 1 yards (standard is 10). Note: Play started inside the 15-yard line (V01).

------------------------------------------------------------
**Total plays reviewed**: 169

In [149]:
# Quick Debug Snippet: Inspect Play Structure for Fumble & Team Attribution
import xml.etree.ElementTree as ET

print("--- DEBUGGING PLAY ATTRIBUTION ---")
found_debug = False

for qtr in root.findall('.//qtr'):
    for play in qtr.findall('play'):
        text = play.get('text', '')
        text_lower = text.lower()
        playid = play.get('playid', '')

        # Look for plays mentioning fumble and team/tm
        if 'fumble' in text_lower and ('team' in text_lower or 'tm' in text_lower):
            print(f"\nFound Play ID: {playid}")
            print(f"Play Text: {text}")
            print(f"Play Attributes: {play.attrib}")
            print("Nested Child Elements & Attributes:")
            for child in play.iter():
                if child != play:
                    print(f"  -> Tag: {child.tag} | Attributes: {child.attrib}")
            found_debug = True

if not found_debug:
    print("No matching play found with 'fumble' and 'team/tm' in text.")

--- DEBUGGING PLAY ATTRIBUTION ---

Found Play ID: 7,4,53
Play Text: Joey Edlington rush for no gain to the LAURIER51, fumble forced by TEAM, fumble by Joey Edlington recovered by LAURIER Marcus Reypa at LAURIER20, clock 10:06.
Play Attributes: {'context': 'H,3,9,H51', 'spot': 'LAURIER51', 'down': '3', 'togo': '9', 'hasball': 'LAURIER', 'text': 'Joey Edlington rush for no gain to the LAURIER51, fumble forced by TEAM, fumble by Joey Edlington recovered by LAURIER Marcus Reypa at LAURIER20, clock 10:06.', 'newcontext': 'V,1,10,H20', 'type': 'R', 'clock': '10:06', 'playid': '7,4,53'}
Nested Child Elements & Attributes:
  -> Tag: p_ru | Attributes: {'gain': '-31', 'vh': 'H', 'name': 'Joey Edlington'}
  -> Tag: p_fumb | Attributes: {'at': 'H20', 'frname': 'Marcus Reypa', 'frvh': 'H', 'name': 'Joey Edlington', 'vh': 'H'}
  -> Tag: p_tk | Attributes: {'ff': 'Y', 'vh': 'V', 'name': 'TEAM'}
